# Face Detection using Haar Cascade (OpenCV)

This notebook demonstrates face (and eye) detection using OpenCV's pre-trained Haar Cascade classifiers, based on the Viola-Jones framework.

## 1. Install and Import Libraries
We need `opencv-python` for Haar Cascade classifiers and `matplotlib` to display images (since `cv2.imshow` doesn't work well in Jupyter).

In [ ]:
# !pip install opencv-python matplotlib --quiet

import cv2
import matplotlib.pyplot as plt

## 2. Load the Pre-trained Haar Cascade Classifier
OpenCV ships with several pre-trained XML cascade files (trained on thousands of Haar features using AdaBoost). We'll use the frontal face and eye cascades.

In [ ]:
# Load pre-trained cascades bundled with OpenCV
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
eye_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_eye.xml')

print("Face cascade loaded:", not face_cascade.empty())
print("Eye cascade loaded:", not eye_cascade.empty())

## 3. Load and Preprocess the Image
Haar Cascade detection works on grayscale images (intensity contrast is what Haar features measure, so color isn't needed).

In [ ]:
# Replace with the path to your own image
image_path = 'Chapter1/images/barack.webp'

img = cv2.imread(image_path)
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.title("Original Image")
plt.axis('off')
plt.show()

## 4. Detect Faces
`detectMultiScale` slides the Haar features (via the cascade) across the image at multiple scales to find regions matching a face.

Key parameters:
- `scaleFactor`: how much the image size is reduced at each scale (1.1 = 10% reduction per step)
- `minNeighbors`: how many overlapping detections are required to confirm a face (higher = fewer false positives)
- `minSize`: minimum possible face size to detect

In [ ]:
faces = face_cascade.detectMultiScale(
    gray,
    scaleFactor=1.1,
    minNeighbors=5,
    minSize=(30, 30)
)

print(f"Number of faces detected: {len(faces)}")
print("Bounding boxes (x, y, w, h):", faces)

## 5. Draw Bounding Boxes Around Detected Faces

In [ ]:
img_with_boxes = img.copy()

for (x, y, w, h) in faces:
    cv2.rectangle(img_with_boxes, (x, y), (x + w, y + h), (0, 255, 0), 2)

plt.imshow(cv2.cvtColor(img_with_boxes, cv2.COLOR_BGR2RGB))
plt.title("Detected Faces")
plt.axis('off')
plt.show()

## 6. (Optional) Detect Eyes Within Each Detected Face
A common pattern: restrict eye detection to the face's bounding box (region of interest) to reduce false positives elsewhere in the image.

In [ ]:
img_with_eyes = img.copy()

for (x, y, w, h) in faces:
    cv2.rectangle(img_with_eyes, (x, y), (x + w, y + h), (0, 255, 0), 2)
    
    # Region of interest: the face area only
    roi_gray = gray[y:y + h, x:x + w]
    roi_color = img_with_eyes[y:y + h, x:x + w]
    
    eyes = eye_cascade.detectMultiScale(roi_gray, scaleFactor=1.1, minNeighbors=10)
    
    for (ex, ey, ew, eh) in eyes:
        cv2.rectangle(roi_color, (ex, ey), (ex + ew, ey + eh), (255, 0, 0), 2)

plt.imshow(cv2.cvtColor(img_with_eyes, cv2.COLOR_BGR2RGB))
plt.title("Detected Faces and Eyes")
plt.axis('off')
plt.show()

## 7. (Optional) Real-Time Face Detection via Webcam
Run this as a standalone Python script (not inside Jupyter) since it opens a live video window.

In [ ]:
# Run as a .py script, not in a notebook cell

# cap = cv2.VideoCapture(0)
#
# while True:
#     ret, frame = cap.read()
#     if not ret:
#         break
#     gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
#     faces = face_cascade.detectMultiScale(gray_frame, scaleFactor=1.1, minNeighbors=5)
#
#     for (x, y, w, h) in faces:
#         cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)
#
#     cv2.imshow('Face Detection', frame)
#     if cv2.waitKey(1) & 0xFF == ord('q'):
#         break
#
# cap.release()
# cv2.destroyAllWindows()



| Step | What Happens |
|---|---|
| Load cascade | Pre-trained Haar features + AdaBoost-selected classifiers (XML file) |
| Convert to grayscale | Haar features rely on intensity contrast, not color |
| `detectMultiScale` | Slides cascade classifier across image at multiple scales |
| `minNeighbors` | Controls strictness of detection (filters false positives) |
| Bounding boxes | Returned as (x, y, width, height) for each detected face |

**Key takeaway**: The XML cascade file already encodes the selected Haar features and cascade stages from training (via AdaBoost on thousands of labeled face/non-face images) — at inference time, we're just applying that pre-trained cascade, not computing Haar features from scratch ourselves.